# 09b Weather Emulator Alpha Sensitivity

This notebook evaluates how the M2 medium-range weather signal changes when the h=3..10 synthetic emulator places less weight on realised target-day weather. The analysis is restricted to M2 and compares the baseline alpha schedule, a stronger drift toward climatology, and two climatology-only variants.

The h=3..10 emulator can be written as a centre plus calibrated forecast error:

```
center(h) = alpha(h) * realised_target_day_weather + (1 - alpha(h)) * ERA5_climatology
point(h)  = center(h) + bias + z * sigma
```

The scenarios are:

- `baseline_alpha_schedule`: the notebook 04 baseline schedule with `alpha(10) = 0.50`, read without overwriting the baseline parquet.
- `stronger_climatology_drift`: the same alpha shape with `alpha(10) = 0.25`.
- `climatology_only_no_noise`: `alpha(h) = 0` for h >= 3, same-calendar-day ERA5 climatology as the centre, and no forecast noise. This is the main low-information benchmark.
- `climatology_only_with_noise`: `alpha(h) = 0` for h >= 3, same-calendar-day ERA5 climatology as the centre, and calibrated forecast-like noise. This is a secondary robustness check.

Scenario weather artifacts and M2-only sensitivity outputs are written under `outputs/weather_emulator_alpha_sensitivity/m2_alpha_sensitivity/`. The baseline parquet and notebook 09 outputs are not overwritten.


## Configuration

The configuration defines the scenario table, origin schedule, horizons, feature sets and output suffixes. Diagnostic-only and notebook-only modes avoid ML fitting; smoke and full modes run M2-only sensitivity models over the configured step-5 origin schedule. Variant parquets are written only inside the dedicated sensitivity folder.


In [ ]:
RUN_MODE = "full"
assert RUN_MODE in {"diagnostic_only", "smoke", "full", "notebook_only"}, f"Unknown RUN_MODE: {RUN_MODE!r}"

# Scenario definitions; baseline remains read-only and climatology-only variants differ by noise handling.
SCENARIO_CONFIG_ROWS = [
    {
        "scenario_id": "baseline_alpha_schedule",
        "alpha_schedule_kind": "baseline_exponential",
        "alpha_h10": 0.50,
        "noise_mode": "baseline_noise",
        "use_noise": True,
        "is_main_benchmark": False,
        "notes": "Read-only baseline schedule from notebook 04.",
    },
    {
        "scenario_id": "stronger_climatology_drift",
        "alpha_schedule_kind": "exponential_alpha_h10_0_25",
        "alpha_h10": 0.25,
        "noise_mode": "baseline_noise",
        "use_noise": True,
        "is_main_benchmark": False,
        "notes": "Conservative drift toward same-calendar-day climatology.",
    },
    {
        "scenario_id": "climatology_only_no_noise",
        "alpha_schedule_kind": "zero_alpha_h3_h10",
        "alpha_h10": 0.00,
        "noise_mode": "no_noise",
        "use_noise": False,
        "is_main_benchmark": True,
        "notes": "Main low-information benchmark: same-calendar-day climatology center, no forecast noise.",
    },
    {
        "scenario_id": "climatology_only_with_noise",
        "alpha_schedule_kind": "zero_alpha_h3_h10",
        "alpha_h10": 0.00,
        "noise_mode": "baseline_noise",
        "use_noise": True,
        "is_main_benchmark": False,
        "notes": "Secondary robustness/stress-test: same-calendar-day climatology center with calibrated forecast-like noise.",
    },
]
SCENARIOS = [row["scenario_id"] for row in SCENARIO_CONFIG_ROWS]
SCENARIO_ALPHA_H10 = {row["scenario_id"]: row["alpha_h10"] for row in SCENARIO_CONFIG_ROWS}
SCENARIO_NOISE_MODE = {row["scenario_id"]: row["noise_mode"] for row in SCENARIO_CONFIG_ROWS}
SCENARIO_SUFFIX = {
    "baseline_alpha_schedule": "",  # no suffix, never overwritten
    "stronger_climatology_drift": "__stronger_climatology_drift",
    "climatology_only_no_noise": "__climatology_only_no_noise",
    "climatology_only_with_noise": "__climatology_only_with_noise",
}

VARIABLES = ["temp", "wind", "humid", "cloud", "precip"]
HORIZONS_DIAGNOSTIC = list(range(0, 11))

# ML sensitivity schedule; smoke and full modes use step-5 origins.
ORIGIN_SCHEDULE_MODE = "step"
ALLOWED_ORIGIN_SCHEDULE_MODES = {"step", "daily", "balanced_weekday"}
assert ORIGIN_SCHEDULE_MODE in ALLOWED_ORIGIN_SCHEDULE_MODES, (
    f"Unknown ORIGIN_SCHEDULE_MODE: {ORIGIN_SCHEDULE_MODE!r}"
)
ORIGIN_STEP_DAYS = 5
EVAL_START_DATE = "2024-08-01"
EVAL_END_DATE = "2025-07-31"
EST_MINUTES_PER_FIT = 0.56
REQUIRE_PREFLIGHT_PASS_FOR_FULL = True
ALLOW_NON_STEP5_FULL = False

SMOKE_N_ORIGINS = 12
SMOKE_HORIZONS = [3, 7, 10]
SMOKE_FEATURE_SETS = ["M2"]
FULL_MAX_ORIGINS = None
FULL_HORIZONS = list(range(3, 11))
FULL_FEATURE_SETS = ["M2"]

if RUN_MODE == "smoke":
    ML_HORIZONS = SMOKE_HORIZONS
    ML_FEATURE_SETS = SMOKE_FEATURE_SETS
    ML_MAX_ORIGINS = SMOKE_N_ORIGINS
elif RUN_MODE == "full":
    ML_HORIZONS = FULL_HORIZONS
    ML_FEATURE_SETS = FULL_FEATURE_SETS
    ML_MAX_ORIGINS = FULL_MAX_ORIGINS
else:
    ML_HORIZONS = []
    ML_FEATURE_SETS = []
    ML_MAX_ORIGINS = None

ML_FILE_SUFFIX = (
    f"_{RUN_MODE}_step{ORIGIN_STEP_DAYS}"
    if RUN_MODE in {"smoke", "full"} and ORIGIN_SCHEDULE_MODE == "step"
    else f"_{RUN_MODE}_{ORIGIN_SCHEDULE_MODE}"
)
RANDOM_STATE = 42
N_JOBS = -1
CLIP_NEGATIVE_PREDICTIONS = True

# Variant parquets stay inside the sensitivity folder and never replace baseline outputs.
FORCE_REBUILD_VARIANTS = True


## Environment and paths

This cell resolves project paths, checks required upstream artifacts, records optional inputs, and writes the scenario configuration used by downstream diagnostics and sensitivity runs.


In [ ]:
import gc
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

MARKER_FILES = ("README_FOR_CODEX.md", "AGENTS.md", "pyproject.toml")


def find_project_root(start=None):
    current = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if any((candidate / marker).exists() for marker in MARKER_FILES):
            return candidate
    raise FileNotFoundError("Could not find the project root. Start Jupyter from inside the project folder.")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_SENS_DIR = OUTPUT_DIR / "weather_emulator_alpha_sensitivity"
M2_ALPHA_SENS_DIR = OUTPUT_SENS_DIR / "m2_alpha_sensitivity"
FIG_DIR = M2_ALPHA_SENS_DIR / "figures"
PREFLIGHT_AUDIT_DIR = M2_ALPHA_SENS_DIR / "preflight_audit"

OUTPUT_SENS_DIR.mkdir(parents=True, exist_ok=True)
M2_ALPHA_SENS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
PREFLIGHT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

# Upstream inputs; these paths are read but not overwritten.
WEATHER_OPERATIONAL_BASELINE_PATH = PROCESSED_DIR / "weather_forecast_operational_windows.parquet"
WEATHER_OPERATIONAL_ML_FEATURES_PATH = PROCESSED_DIR / "weather_forecast_operational_ml_features.parquet"
REALISED_WEATHER_WINDOWS_PATH = PROCESSED_DIR / "realised_weather_daily_windows.parquet"
ERA5_SAME_CALENDAR_DAY_CLIMATOLOGY_PATH = PROCESSED_DIR / "era5_longrun_same_calendar_day_climatology.parquet"
ERA5_CLIMATOLOGY_PARAMETERS_PATH = PROCESSED_DIR / "era5_longrun_climatology_parameters.parquet"
ML_FORECAST_PANEL_PATH = PROCESSED_DIR / "ml_forecast_panel_full.parquet"
ERA5_CLIMATOLOGY_USAGE_SUMMARY_PATH = OUTPUT_DIR / "synthetic_weather_model" / "era5_climatology_usage_summary.csv"
CLIMATOLOGY_DRIFT_ALPHA_SCHEDULE_PATH = OUTPUT_DIR / "synthetic_weather_model" / "climatology_drift_alpha_schedule.csv"
TUNING_BEST_PARAMS_PATH = OUTPUT_DIR / "lightgbm_tuning" / "lightgbm_optuna_best_params_full.json"
M1_FULL_STEP5_METRICS_PATH = OUTPUT_DIR / "lightgbm_rolling_origin" / "lightgbm_rolling_origin_metrics_full_step5.csv"


def project_relative(path):
    path = Path(path)
    try:
        return path.relative_to(PROJECT_ROOT).as_posix()
    except ValueError:
        return str(path)


def require_file(path, description):
    if not Path(path).exists():
        raise FileNotFoundError(
            f"Missing {description}: {project_relative(path)}. "
            "Rerun the producing notebook before running this notebook."
        )


REQUIRED_INPUTS = [
    (WEATHER_OPERATIONAL_BASELINE_PATH, "baseline operational weather windows (notebook 04)"),
    (REALISED_WEATHER_WINDOWS_PATH, "realised weather windows (notebook 01a)"),
    (ERA5_SAME_CALENDAR_DAY_CLIMATOLOGY_PATH, "ERA5 same-calendar-day climatology (notebook 01e)"),
    (ERA5_CLIMATOLOGY_PARAMETERS_PATH, "ERA5 long-run fallback climatology parameters (notebook 01e)"),
    (ML_FORECAST_PANEL_PATH, "ML forecast panel (notebook 06)"),
]
OPTIONAL_INPUTS = [
    (WEATHER_OPERATIONAL_ML_FEATURES_PATH, "ML-ready weather feature view (notebook 05)"),
    (CLIMATOLOGY_DRIFT_ALPHA_SCHEDULE_PATH, "alpha schedule audit (notebook 04)"),
    (ERA5_CLIMATOLOGY_USAGE_SUMMARY_PATH, "ERA5 climatology usage summary (notebook 04)"),
    (TUNING_BEST_PARAMS_PATH, "tuned LightGBM parameters (notebook 08)"),
    (M1_FULL_STEP5_METRICS_PATH, "existing full_step5 M1 reference metrics (notebook 09)"),
]
for path, desc in REQUIRED_INPUTS:
    require_file(path, desc)
for path, desc in OPTIONAL_INPUTS:
    if path.exists():
        print(f"  optional input present: {project_relative(path)}")
    else:
        print(f"  optional input MISSING (some downstream cells will skip): {project_relative(path)}")

SCENARIO_CONFIG = pd.DataFrame(SCENARIO_CONFIG_ROWS)
SCENARIO_CONFIG_PATH = M2_ALPHA_SENS_DIR / "m2_alpha_sensitivity_scenario_config.csv"
SCENARIO_CONFIG.to_csv(SCENARIO_CONFIG_PATH, index=False)
OPERATIONAL_VARIANT_PATHS = {
    sc: (
        WEATHER_OPERATIONAL_BASELINE_PATH
        if sc == "baseline_alpha_schedule"
        else M2_ALPHA_SENS_DIR / f"weather_forecast_operational_windows{SCENARIO_SUFFIX[sc]}.parquet"
    )
    for sc in SCENARIOS
}
print()
print("Variant operational weather parquet paths:")
for sc, p in OPERATIONAL_VARIANT_PATHS.items():
    print(f"  {sc}: {project_relative(p)}")
print()
print(f"RUN_MODE: {RUN_MODE}")
print(f"Origin mode: {ORIGIN_SCHEDULE_MODE}")
print(f"Origin step days: {ORIGIN_STEP_DAYS if ORIGIN_SCHEDULE_MODE == 'step' else 'n/a'}")
print(f"ML file suffix: {ML_FILE_SUFFIX}")
print(f"SCENARIOS: {SCENARIOS}")
print(f"SCENARIO_ALPHA_H10: {SCENARIO_ALPHA_H10}")
print(f"Scenario config: {project_relative(SCENARIO_CONFIG_PATH)}")
print(f"Sensitivity output folder: {project_relative(M2_ALPHA_SENS_DIR)}")


## Baseline alpha audit

This cell reconstructs the baseline and scenario alpha schedules, writes the alpha schedule table, and audits the operational weather schemas for realised-weather, climatology-value and pressure-like columns.


In [ ]:
def baseline_alpha_for_horizon(h, alpha_at_max=0.50, max_h=10):
    """Reproduce the notebook 04 formula: smooth exponential decay with floor."""
    h = np.asarray(h, dtype="float64")
    horizon_span = max(float(max_h - 2), 1.0)
    lam = -np.log(float(alpha_at_max)) / horizon_span
    alpha = np.exp(-lam * np.clip(h - 2.0, 0.0, None))
    return np.clip(alpha, float(alpha_at_max), 1.0)


def alpha_for_scenario(h, scenario):
    """Per-scenario alpha schedule. h=0-h=2 deterministic MEPS rows are kept
    unchanged in all scenarios; alpha is defined as 1.0 there but never applied
    because the baseline parquet already contains the MEPS forecast values."""
    h_arr = np.asarray(h, dtype="float64")
    alpha_at_max = SCENARIO_ALPHA_H10[scenario]
    if scenario in {"climatology_only_no_noise", "climatology_only_with_noise"}:
        # h=0..2 deterministic rows are retained; alpha=0 is applied only for h>=3.
        alpha = np.where(h_arr <= 2.0, 1.0, 0.0)
    else:
        alpha = baseline_alpha_for_horizon(h_arr, alpha_at_max=alpha_at_max, max_h=10)
    return alpha.astype("float64")


# Scenario alpha endpoints.
assert abs(float(baseline_alpha_for_horizon(10)) - 0.50) < 1e-6, "baseline alpha(10) must be 0.50"
assert abs(float(alpha_for_scenario(10, "baseline_alpha_schedule")) - 0.50) < 1e-6
assert abs(float(alpha_for_scenario(10, "stronger_climatology_drift")) - 0.25) < 1e-6
assert abs(float(alpha_for_scenario(10, "climatology_only_no_noise")) - 0.0) < 1e-12
assert abs(float(alpha_for_scenario(10, "climatology_only_with_noise")) - 0.0) < 1e-12

# Schedule table for diagnostics and documentation.
schedule_rows = []
for h in HORIZONS_DIAGNOSTIC:
    for sc in SCENARIOS:
        schedule_rows.append(
            {
                "scenario": sc,
                "scenario_alpha_at_h10": SCENARIO_ALPHA_H10[sc],
                "noise_mode": SCENARIO_NOISE_MODE[sc],
                "horizon": int(h),
                "alpha": float(alpha_for_scenario(h, sc)),
                "climatology_weight": float(1.0 - alpha_for_scenario(h, sc)),
                "horizon_tier": "actual_meps_h0_h2" if h <= 2 else "synthetic_scenario_h3_h10",
            }
        )
alpha_schedules_df = pd.DataFrame(schedule_rows)
ALPHA_SCHEDULES_PATH = M2_ALPHA_SENS_DIR / "m2_alpha_sensitivity_alpha_schedules.csv"
alpha_schedules_df.to_csv(ALPHA_SCHEDULES_PATH, index=False)
print(f"Saved: {project_relative(ALPHA_SCHEDULES_PATH)}")

# Schema audit for operational weather columns.
# Read schema only.
baseline_schema = pq.ParquetFile(WEATHER_OPERATIONAL_BASELINE_PATH).schema_arrow.names

PRESSURE_NAME_PATTERNS = ("msl", "pressure", "apparent")
OBS_TRUTH_PATTERNS = ("_obs_", "obs_", "_truth", "_realised", "realised_")
CLIM_VALUE_PATTERNS = ("_clim_", "clim_value", "climatology_value")

operational_pressure_cols = [c for c in baseline_schema if any(s in c.lower() for s in PRESSURE_NAME_PATTERNS)]
operational_obs_truth_cols = [c for c in baseline_schema if any(s in c.lower() for s in OBS_TRUTH_PATTERNS)]
operational_clim_value_cols = [c for c in baseline_schema if any(s in c.lower() for s in CLIM_VALUE_PATTERNS)]

print()
print("Baseline operational weather parquet schema audit:")
print(f"  rows in schema: {pq.ParquetFile(WEATHER_OPERATIONAL_BASELINE_PATH).metadata.num_rows:,}")
print(f"  columns: {len(baseline_schema)}")
print(f"  pressure-like columns: {operational_pressure_cols}")
print(f"  observed/truth columns:  {operational_obs_truth_cols}")
print(f"  climatology-value columns: {operational_clim_value_cols}")
print("  (The first two should be empty in the operational parquet: ")
print("   the baseline emulator never exposes them downstream as ML features.)")

# Audit the ML-ready weather view when available.
if WEATHER_OPERATIONAL_ML_FEATURES_PATH.exists():
    ml05_schema = pq.ParquetFile(WEATHER_OPERATIONAL_ML_FEATURES_PATH).schema_arrow.names
    ml05_pressure = [c for c in ml05_schema if any(s in c.lower() for s in PRESSURE_NAME_PATTERNS)]
    ml05_obs = [c for c in ml05_schema if any(s in c.lower() for s in OBS_TRUTH_PATTERNS)]
    ml05_clim = [c for c in ml05_schema if any(s in c.lower() for s in CLIM_VALUE_PATTERNS)]
    print()
    print("ML feature view (notebook 05) schema audit:")
    print(f"  pressure-like columns: {ml05_pressure}")
    print(f"  observed/truth columns:  {ml05_obs}")
    print(f"  climatology-value columns: {ml05_clim}")


## Variant weather forecast construction

Scenario variants are constructed by shifting the forecast centre relative to the baseline emulator. For a new alpha schedule, the point forecast is obtained by adding `(alpha_new(h) - alpha_baseline(h)) * (truth - climatology)` to the baseline forecast.

For `baseline_noise` scenarios, this preserves the calibrated residual draw from the baseline. For `no_noise`, h=3..10 rows are set directly to the scenario centre. Physical bounds are reapplied after the shift, and same-calendar-day ERA5 climatology is used as the primary climatology source with legacy monthly or seasonal levels retained only as fallback.


In [ ]:
POINT_COLUMNS = {
    "temp": "temp_fcst_mean",
    "wind": "wind_fcst_mean",
    "humid": "humid_fcst_mean",
    "cloud":  "cloud_fcst_mean",
    "precip": "precip_fcst_sum",
}
P10_COLUMNS = {v: f"{v}_fcst_p10" for v in POINT_COLUMNS}
P90_COLUMNS = {v: f"{v}_fcst_p90" for v in POINT_COLUMNS}
# Precipitation quantile names use the explicit forecast prefix.
P10_COLUMNS["precip"] = "precip_fcst_p10"
P90_COLUMNS["precip"] = "precip_fcst_p90"

PHYSICAL_BOUNDS = {
    "temp": (None, None),  # bounded only via empirical bounds; left unclipped here
    "wind": (0.0, None),
    "humid": (0.0, 100.0),
    "cloud": (0.0, 100.0),
    "precip": (0.0, None),
}

# Same-calendar-day climatology should cover nearly all AvdelingID/window/month/day rows.
EMERGENCY_FALLBACK_MAX_SHARE = 0.01


def realised_value_columns(variable):
    """Map variable to its realised window column in realised_weather_daily_windows.parquet."""
    return {
        "temp": "temp_obs_mean",
        "wind": "wind_obs_mean",
        "humid": "humid_obs_mean",
        "cloud":  "cloud_obs_mean",
        "precip": "precip_obs_sum",
    }[variable]


def era5_clim_columns(variable):
    """Map variable to the ERA5 climatology mean column.

    For precipitation, the operational emulator uses precip_wet_amount_mean as climatology
    (conditional on wet days); we adopt the same convention here to match notebook 04.
    """
    return {
        "temp": "temp_clim_mean",
        "wind": "wind_clim_mean",
        "humid": "humid_clim_mean",
        "cloud":  "cloud_clim_mean",
        "precip": "precip_wet_amount_mean",
    }[variable]


def load_realised_window(window: str = "trade_08_19") -> pd.DataFrame:
    """Load the realised window table for the selected operational window."""
    real = pd.read_parquet(REALISED_WEATHER_WINDOWS_PATH)
    real = real.loc[real["aggregation_window"] == window].copy()
    real["date"] = pd.to_datetime(real["date"]).dt.normalize()
    return real


def load_era5_same_calendar_day_climatology(window: str = "trade_08_19") -> pd.DataFrame:
    """Primary ERA5 climatology keyed by AvdelingID/window/month/day, years 2000-2019."""
    era5 = pd.read_parquet(ERA5_SAME_CALENDAR_DAY_CLIMATOLOGY_PATH)
    era5 = era5.loc[era5["aggregation_window"] == window].copy()
    return era5


def load_era5_fallback_climatology(window: str = "trade_08_19") -> pd.DataFrame:
    """Legacy monthly/seasonal ERA5 parameters retained as fallback only."""
    era5 = pd.read_parquet(ERA5_CLIMATOLOGY_PARAMETERS_PATH)
    era5 = era5.loc[era5["aggregation_window"] == window].copy()
    return era5


def validate_era5_same_day_contract():
    """Phase 3: validate the corrected ERA5 climatology input contract before any heavy build.

    Requires the AvdelingID-keyed same-calendar-day climatology from the patched notebook 01e.
    Raises RuntimeError (never silently falls back to store_point-only climatology) so that a stale
    or mis-keyed climatology cannot quietly degrade the alpha-sensitivity scenarios.
    """
    same_day_path = ERA5_SAME_CALENDAR_DAY_CLIMATOLOGY_PATH
    params_path = ERA5_CLIMATOLOGY_PARAMETERS_PATH
    if not Path(same_day_path).exists():
        raise RuntimeError(
            f"Missing same-calendar-day ERA5 climatology: {project_relative(same_day_path)}; "
            "rerun notebook 01e."
        )
    if not Path(params_path).exists():
        raise RuntimeError(
            f"Missing ERA5 fallback climatology parameters: {project_relative(params_path)}; "
            "rerun notebook 01e."
        )
    schema_names = list(pq.read_schema(same_day_path).names)
    required = ["AvdelingID", "aggregation_window", "month", "day"]
    missing = [c for c in required if c not in schema_names]
    if missing:
        raise RuntimeError(
            f"Same-calendar-day climatology is missing required key columns {missing}; found {schema_names}. "
            "Rerun the patched notebook 01e so it emits AvdelingID-keyed climatology."
        )
    has_store_point = "store_point" in schema_names
    read_cols = required + [c for c in ["climatology_location_key_column", "climatology_definition"] if c in schema_names]
    df = pd.read_parquet(same_day_path, columns=read_cols)
    if pd.to_numeric(df["AvdelingID"], errors="coerce").notna().sum() == 0:
        raise RuntimeError(
            "Same-calendar-day climatology AvdelingID column is entirely missing/non-numeric; "
            "expected an AvdelingID-keyed climatology (rerun the patched notebook 01e)."
        )
    if "climatology_location_key_column" in df.columns:
        key_vals = set(pd.Series(df["climatology_location_key_column"]).dropna().astype(str).unique())
        if key_vals and key_vals != {"AvdelingID"}:
            raise RuntimeError(
                f"Same-calendar-day climatology_location_key_column must be 'AvdelingID'; found {sorted(key_vals)}. "
                "Refusing to fall back to store_point-only climatology in 09b."
            )
    if "climatology_definition" in df.columns:
        defs = set(pd.Series(df["climatology_definition"]).dropna().astype(str).unique())
        expected_def = "same_calendar_day_avdelingid_window_2000_2019"
        if defs and expected_def not in defs:
            raise RuntimeError(
                f"Same-calendar-day climatology_definition must include {expected_def!r}; found {sorted(defs)}."
            )
    dup = int(df.duplicated(["AvdelingID", "aggregation_window", "month", "day"]).sum())
    if dup > 0:
        raise RuntimeError(
            f"Same-calendar-day climatology has {dup} duplicate rows on [AvdelingID, aggregation_window, month, day]; "
            "this would create a many-to-one merge ambiguity. Rerun notebook 01e."
        )
    return {"rows": int(len(df)), "has_store_point": bool(has_store_point), "duplicates": dup}


def as_float_array(series_or_array, copy=True):
    """Return a float64 numpy array. Use copy=True for arrays that will be mutated.

    Pandas/PyArrow-backed columns can return read-only numpy views from .to_numpy(); masked
    in-place assignment into such a view raises 'assignment destination is read-only'. copy=True
    guarantees a writable array. For arrays only read (e.g. fed to np.where) copy=False is fine.
    """
    return pd.to_numeric(series_or_array, errors="coerce").to_numpy(dtype="float64", copy=copy)


def build_truth_clim_table(window: str = "trade_08_19") -> pd.DataFrame:
    """Build the per-row (target_date, AvdelingID, window) table of realised
    truth and ERA5 climatology for each variable."""
    # Validate the AvdelingID-keyed same-day climatology contract before merging.
    validate_era5_same_day_contract()

    real = load_realised_window(window)
    real = real.rename(columns={"date": "target_date"})
    era5_same_day = load_era5_same_calendar_day_climatology(window)
    era5 = load_era5_fallback_climatology(window)

    # Legacy ERA5 fallback levels are encoded as integer codes 1..5:
    #   1 = store_month_window, 2 = store_season_window, 3 = month_window,
    #   4 = season_window, 5 = window.
    fallback_priority = [
        (0, "store_same_calendar_day_window"),
        (1, "store_month_window"),
        (2, "store_season_window"),
        (3, "month_window"),
        (4, "season_window"),
        (5, "window"),
    ]
    # Coerce AvdelingID to a common numeric dtype before merging.
    era5_same_day["AvdelingID"] = pd.to_numeric(era5_same_day["AvdelingID"], errors="coerce")
    era5["AvdelingID"] = pd.to_numeric(era5["AvdelingID"], errors="coerce")
    real["AvdelingID"] = pd.to_numeric(real["AvdelingID"], errors="coerce")
    era5_by_level = {label: era5.loc[era5["fallback_level"] == code].copy()
                     for code, label in fallback_priority if code != 0}
    era5_by_level["store_same_calendar_day_window"] = era5_same_day.copy()

    # Add calendar keys for same-day and fallback climatology.
    real["target_month"] = real["target_date"].dt.month.astype("int8")
    real["target_day"] = real["target_date"].dt.day.astype("int8")
    season_map = {12: "winter", 1: "winter", 2: "winter",
                  3: "spring", 4: "spring", 5: "spring",
                  6: "summer", 7: "summer", 8: "summer",
                  9: "autumn", 10: "autumn", 11: "autumn"}
    real["target_season"] = real["target_month"].map(season_map)

    truth_cols = ["target_date", "AvdelingID", "aggregation_window", "target_month", "target_day", "target_season"]
    for v in VARIABLES:
        truth_cols.append(realised_value_columns(v))
    truth_df = real[truth_cols].copy()

    # Resolve climatology through same-day primary and fallback levels.
    out = truth_df.copy()
    for v in VARIABLES:
        out[f"{v}_clim_value"] = np.nan
        out[f"clim_fallback_level_{v}"] = pd.Series([None] * len(out), dtype="object")

    for _code, fl in fallback_priority:
        block = era5_by_level[fl]
        clim_cols = [era5_clim_columns(v) for v in VARIABLES]
        keep = [c for c in clim_cols if c in block.columns]
        if not keep:
            continue
        # Build merge keys for the active fallback level.
        if fl == "store_same_calendar_day_window":
            on = ["AvdelingID", "target_month", "target_day", "aggregation_window"]
            block_keys = block[["AvdelingID", "month", "day", "aggregation_window"] + keep].rename(
                columns={"month": "target_month", "day": "target_day"}
            )
        elif fl == "store_month_window":
            on = ["AvdelingID", "target_month", "aggregation_window"]
            block_keys = block[["AvdelingID", "month", "aggregation_window"] + keep].rename(
                columns={"month": "target_month"}
            )
        elif fl == "store_season_window":
            on = ["AvdelingID", "target_season", "aggregation_window"]
            block_keys = block[["AvdelingID", "season", "aggregation_window"] + keep].rename(
                columns={"season": "target_season"}
            )
        elif fl == "month_window":
            on = ["target_month", "aggregation_window"]
            block_keys = block[["month", "aggregation_window"] + keep].rename(
                columns={"month": "target_month"}
            )
        elif fl == "season_window":
            on = ["target_season", "aggregation_window"]
            block_keys = block[["season", "aggregation_window"] + keep].rename(
                columns={"season": "target_season"}
            )
        else:  # window
            on = ["aggregation_window"]
            block_keys = block[["aggregation_window"] + keep]
        # Refuse many-to-one merge ambiguity at any fallback level.
        block_keys = block_keys.dropna(subset=on)
        dup_keys = int(block_keys.duplicated(on).sum())
        if dup_keys > 0:
            raise RuntimeError(
                f"Climatology fallback level {fl!r} has {dup_keys} duplicate merge keys on {on}; "
                "refusing an ambiguous many-to-one merge."
            )
        rows_before = len(out)
        merged = out.merge(block_keys, on=on, how="left", suffixes=("", f"__{fl}"))
        if len(merged) != rows_before:
            raise RuntimeError(
                f"Climatology fallback level {fl!r} changed row count {rows_before} -> {len(merged)} "
                f"(unexpected many-to-many merge on {on})."
            )
        for v in VARIABLES:
            era5_col = era5_clim_columns(v)
            cand_col = f"{era5_col}__{fl}" if f"{era5_col}__{fl}" in merged.columns else era5_col
            if cand_col in merged.columns:
                # Writable copies are required because current arrays are mutated below.
                fill = as_float_array(merged[cand_col], copy=False)
                cur = as_float_array(out[f"{v}_clim_value"], copy=True)
                # Fill missing climatology values at this fallback level.
                mask = np.isnan(cur) & ~np.isnan(fill)
                cur[mask] = fill[mask]
                out[f"{v}_clim_value"] = cur
                # Track the fallback level supplying each climatology value.
                level_col = out[f"clim_fallback_level_{v}"].to_numpy(dtype="object", copy=True)
                level_col[mask] = fl
                out[f"clim_fallback_level_{v}"] = level_col

    # Last-resort fallback uses realised truth only when climatology is unavailable.
    for v in VARIABLES:
        clim = as_float_array(out[f"{v}_clim_value"], copy=True)
        truth = as_float_array(out[realised_value_columns(v)], copy=False)
        mask = np.isnan(clim) & ~np.isnan(truth)
        clim[mask] = truth[mask]
        out[f"{v}_clim_value"] = clim
        level_col = out[f"clim_fallback_level_{v}"].to_numpy(dtype="object", copy=True)
        level_col[mask] = "realised_truth_emergency"
        out[f"clim_fallback_level_{v}"] = level_col

    # Guard against excessive emergency fallback use.
    coverage_rows = []
    emergency_problems = []
    for v in VARIABLES:
        lv = out[f"clim_fallback_level_{v}"]
        same_day_share = float((lv == "store_same_calendar_day_window").mean()) if len(out) else float("nan")
        emergency_share = float((lv == "realised_truth_emergency").mean()) if len(out) else 0.0
        coverage_rows.append({
            "variable": v,
            "same_calendar_day_share": same_day_share,
            "emergency_truth_share": emergency_share,
            "n_rows": int(len(out)),
        })
        if emergency_share > EMERGENCY_FALLBACK_MAX_SHARE:
            emergency_problems.append(f"{v}={emergency_share:.4f}")
    out.attrs["clim_coverage"] = pd.DataFrame(coverage_rows)
    if emergency_problems:
        raise RuntimeError(
            "Realised-truth emergency climatology fallback exceeds "
            f"{EMERGENCY_FALLBACK_MAX_SHARE:.2%} for: {', '.join(emergency_problems)}. "
            "Same-calendar-day climatology should cover almost all rows after the 01e/04 fix; "
            "investigate the climatology merge before trusting the alpha scenarios."
        )

    out["AvdelingID"] = pd.to_numeric(out["AvdelingID"], errors="coerce")
    return out


# Build realised and climatology columns once per run.
print("Building truth + ERA5 climatology lookup ...")
t0 = time.time()
TRUTH_CLIM = build_truth_clim_table(window="trade_08_19") if RUN_MODE != "notebook_only" else pd.DataFrame()
print(f"  done in {time.time() - t0:.1f}s; rows: {len(TRUTH_CLIM):,}")


def print_preflight_summary():
    """Phase 7: cheap go/no-go summary printed before any heavy variant/ML work."""
    print("=" * 72)
    print("09b ALPHA-SENSITIVITY PREFLIGHT SUMMARY")
    print("=" * 72)
    print(f"RUN_MODE: {RUN_MODE}")
    print(f"Scenarios: {SCENARIOS}")
    print("Main low-information benchmark: climatology_only_no_noise (secondary: climatology_only_with_noise)")
    try:
        contract = validate_era5_same_day_contract()
        print(
            f"Same-day ERA5 climatology: AvdelingID-keyed OK; rows={contract['rows']:,}; "
            f"store_point helper present={contract['has_store_point']}; duplicate keys={contract['duplicates']}"
        )
    except Exception as exc:
        print(f"Same-day ERA5 climatology contract CHECK FAILED: {exc}")
        raise
    if RUN_MODE == "notebook_only":
        print("TRUTH_CLIM rows: 0 (not built in notebook_only mode)")
    else:
        print(f"TRUTH_CLIM rows: {len(TRUTH_CLIM):,}")
        cov = TRUTH_CLIM.attrs.get("clim_coverage") if hasattr(TRUTH_CLIM, "attrs") else None
        if cov is not None and not cov.empty:
            for _, r in cov.iterrows():
                print(f"  {r['variable']}: same_calendar_day_share={r['same_calendar_day_share']:.4f} "
                      f"emergency_truth_share={r['emergency_truth_share']:.4f}")
    print("Variant operational weather parquet paths (baseline read-only; variants non-overwriting):")
    for sc, p in OPERATIONAL_VARIANT_PATHS.items():
        if sc == "baseline_alpha_schedule":
            status = "baseline (read-only)"
        elif Path(p).exists() and not FORCE_REBUILD_VARIANTS:
            status = "exists -> reuse/skip"
        elif Path(p).exists() and FORCE_REBUILD_VARIANTS:
            status = "exists -> WILL REBUILD (FORCE_REBUILD_VARIANTS=True)"
        else:
            status = "will build"
        print(f"  {sc}: {project_relative(p)} -> {status}")
    print(f"FORCE_REBUILD_VARIANTS: {FORCE_REBUILD_VARIANTS}")
    if RUN_MODE in {"smoke", "full"}:
        n_orig = ML_MAX_ORIGINS if ML_MAX_ORIGINS else "all"
        print(f"Expected ML workload: scenarios={len(SCENARIOS)} x horizons={len(ML_HORIZONS)} x "
              f"feature_sets={len(ML_FEATURE_SETS)} x origins={n_orig} (M2-only); ~{EST_MINUTES_PER_FIT} min/fit.")
    else:
        print("Expected ML workload: none (ML training skipped in this RUN_MODE).")
    print("=" * 72)


print_preflight_summary()


def construct_variant(scenario: str) -> pd.DataFrame:
    """Produce the variant operational weather parquet for the given scenario.

    Reads the baseline parquet, joins truth+clim by (target_date, AvdelingID,
    aggregation_window), applies the closed-form delta shift to the point /
    p10 / p90 columns, reapplies physical bounds, and returns the new frame.
    """
    if scenario == "baseline_alpha_schedule":
        # Baseline scenario is read without writing a new parquet.
        return pd.read_parquet(WEATHER_OPERATIONAL_BASELINE_PATH)

    base = pd.read_parquet(WEATHER_OPERATIONAL_BASELINE_PATH)
    base["target_date"] = pd.to_datetime(base["target_date"]).dt.normalize()
    base["AvdelingID"] = pd.to_numeric(base["AvdelingID"], errors="coerce")

    merge_keys = ["target_date", "AvdelingID", "aggregation_window"]
    joined = base.merge(
        TRUTH_CLIM[merge_keys + [f"{v}_clim_value" for v in VARIABLES]
                   + [realised_value_columns(v) for v in VARIABLES]],
        on=merge_keys,
        how="left",
        validate="m:1",
    )

    h_arr = joined["horizon_days"].astype("float64").to_numpy()
    alpha_baseline = baseline_alpha_for_horizon(h_arr, alpha_at_max=0.50, max_h=10)
    alpha_new = alpha_for_scenario(h_arr, scenario)
    da = (alpha_new - alpha_baseline).astype("float64")  # broadcastable across rows
    noise_mode = SCENARIO_NOISE_MODE[scenario]

    # Apply scenario shifts only to h>=3; h=0..2 deterministic MEPS rows are unchanged.
    synthetic_mask = h_arr >= 3.0

    out = joined.copy()
    out["alpha_scenario"] = alpha_new.astype("float32")
    out["alpha_baseline"] = alpha_baseline.astype("float32")
    out["delta_alpha"] = da.astype("float32")
    out["scenario"] = scenario
    out["noise_mode"] = noise_mode

    bound_share_rows = []
    for v in VARIABLES:
        pc = POINT_COLUMNS[v]
        # These arrays are read-only inputs to np.where.
        truth = as_float_array(out[realised_value_columns(v)], copy=False)
        clim  = as_float_array(out[f"{v}_clim_value"], copy=False)
        gap   = truth - clim
        # baseline_noise preserves the calibrated residual through a centre shift.
        # no_noise sets h>=3 directly to the scenario centre.
        shift = np.where(synthetic_mask, da * gap, 0.0)
        baseline_point = as_float_array(out[pc], copy=False)
        center_new = alpha_new * truth + (1.0 - alpha_new) * clim
        if noise_mode == "no_noise":
            new_point = np.where(synthetic_mask, center_new, baseline_point)
        elif noise_mode == "baseline_noise":
            new_point = baseline_point + shift
        else:
            raise ValueError(f"Unknown noise_mode for {scenario}: {noise_mode!r}")
        lo, hi = PHYSICAL_BOUNDS[v]
        if lo is not None:
            new_point = np.where(new_point < lo, lo, new_point)
        if hi is not None:
            new_point = np.where(new_point > hi, hi, new_point)
        out[pc] = new_point.astype("float32")
        # Quantile columns shift by the same alpha delta, then bounds are reapplied.
        for q_col in (P10_COLUMNS[v], P90_COLUMNS[v]):
            if q_col in out.columns:
                q_base = as_float_array(out[q_col], copy=False)
                q = np.where(synthetic_mask, center_new, q_base) if noise_mode == "no_noise" else q_base + shift
                if lo is not None:
                    q = np.where(q < lo, lo, q)
                if hi is not None:
                    q = np.where(q > hi, hi, q)
                out[q_col] = q.astype("float32")
        # Bound-share audit.
        if lo is not None or hi is not None:
            at_lower = float(np.mean(np.isclose(new_point, lo if lo is not None else -np.inf)))
            at_upper = float(np.mean(np.isclose(new_point, hi if hi is not None else np.inf)))
            bound_share_rows.append({
                "scenario": scenario, "variable": v,
                "share_at_lower_bound": at_lower, "share_at_upper_bound": at_upper,
            })

    # Drop auxiliary truth and climatology columns before writing scenario output.
    aux_cols = [realised_value_columns(v) for v in VARIABLES] + [f"{v}_clim_value" for v in VARIABLES]
    out = out.drop(columns=[c for c in aux_cols if c in out.columns])

    out._bound_share = pd.DataFrame(bound_share_rows)  # attach for diagnostics
    return out


# Generate and persist non-baseline scenario parquets.
if RUN_MODE in {"diagnostic_only", "smoke"}:
    bound_share_blocks = []
    for sc in SCENARIOS:
        if sc == "baseline_alpha_schedule":
            print(f"\nScenario {sc}: read-only, no variant parquet written.")
            continue
        path = OPERATIONAL_VARIANT_PATHS[sc]
        if path.exists() and not FORCE_REBUILD_VARIANTS:
            print(f"\nScenario {sc}: variant parquet already exists at "
                  f"{project_relative(path)}; skipping regeneration "
                  f"(set FORCE_REBUILD_VARIANTS=True or delete the file to force rebuild).")
            continue
        print(f"\nScenario {sc}: building variant parquet at {project_relative(path)} ...")
        t0 = time.time()
        variant = construct_variant(sc)
        print(f"  built in {time.time() - t0:.1f}s; rows: {len(variant):,}")
        # Refuse writes outside the sensitivity folder or to the baseline filename.
        assert path != WEATHER_OPERATIONAL_BASELINE_PATH, "variant path collides with baseline"
        assert Path(path).parent == M2_ALPHA_SENS_DIR, "variant must be written under the sensitivity folder"
        variant.to_parquet(path, index=False)
        if hasattr(variant, "_bound_share") and not variant._bound_share.empty:
            bound_share_blocks.append(variant._bound_share)
        del variant
        gc.collect()
    if bound_share_blocks:
        bound_share_df = pd.concat(bound_share_blocks, ignore_index=True)
        bound_share_df.to_csv(M2_ALPHA_SENS_DIR / "m2_alpha_sensitivity_value_range_checks.csv", index=False)
        print(f"\nSaved: {project_relative(M2_ALPHA_SENS_DIR / 'm2_alpha_sensitivity_value_range_checks.csv')}")
else:
    print("\nRUN_MODE = 'notebook_only': skipping variant construction.")


## Non-ML weather diagnostics

This cell summarizes each scenario by horizon and weather variable for the operational `trade_08_19` window. Diagnostics include forecast-realised association, forecast error against realised weather, distance from climatology, forecast dispersion, physical-bound shares, wet-day probabilities and cloud-regime stability when available.


In [ ]:
def _safe_corr(x, y):
    x = pd.to_numeric(x, errors="coerce")
    y = pd.to_numeric(y, errors="coerce")
    df = pd.DataFrame({"x": x, "y": y}).dropna()
    if len(df) < 5 or df["x"].std() == 0 or df["y"].std() == 0:
        return float("nan")
    xv = df["x"].to_numpy(dtype="float64")
    yv = df["y"].to_numpy(dtype="float64")
    xc = xv - xv.mean()
    yc = yv - yv.mean()
    denom = np.sqrt((xc * xc).sum() * (yc * yc).sum())
    if denom == 0 or not np.isfinite(denom):
        return float("nan")
    return float((xc * yc).sum() / denom)


def diagnostic_rows_for(variant: pd.DataFrame, scenario: str, truth_clim: pd.DataFrame):
    """Compute diagnostics by horizon and variable for one scenario."""
    rows_corr_err = []
    rows_clim_dist = []
    rows_basic = []
    merge_keys = ["target_date", "AvdelingID", "aggregation_window"]
    block = variant.merge(
        truth_clim[merge_keys + [f"{v}_clim_value" for v in VARIABLES]
                   + [realised_value_columns(v) for v in VARIABLES]],
        on=merge_keys,
        how="left",
    )
    block = block.loc[block["aggregation_window"] == "trade_08_19"].reset_index(drop=True)
    for h in HORIZONS_DIAGNOSTIC:
        sub = block.loc[block["horizon_days"] == h]
        if sub.empty:
            continue
        for v in VARIABLES:
            fcst = pd.to_numeric(sub[POINT_COLUMNS[v]], errors="coerce")
            truth = pd.to_numeric(sub[realised_value_columns(v)], errors="coerce")
            clim = pd.to_numeric(sub[f"{v}_clim_value"], errors="coerce")
            diff_truth = fcst - truth
            diff_clim = fcst - clim
            rows_corr_err.append(
                {
                    "scenario": scenario,
                    "horizon": int(h),
                    "variable": v,
                    "n": int(diff_truth.notna().sum()),
                    "corr_forecast_realised": _safe_corr(fcst, truth),
                    "mae_forecast_realised": float(diff_truth.abs().mean(skipna=True)),
                    "rmse_forecast_realised": float(np.sqrt((diff_truth ** 2).mean(skipna=True))),
                }
            )
            rows_clim_dist.append(
                {
                    "scenario": scenario,
                    "horizon": int(h),
                    "variable": v,
                    "n": int(diff_clim.notna().sum()),
                    "mae_forecast_climatology": float(diff_clim.abs().mean(skipna=True)),
                    "rmse_forecast_climatology": float(np.sqrt((diff_clim ** 2).mean(skipna=True))),
                }
            )
            fcst_std = float(fcst.std(skipna=True))
            real_std = float(truth.std(skipna=True))
            ratio = (fcst_std / real_std) if (real_std and not np.isnan(real_std) and real_std > 0) else float("nan")
            rows_basic.append(
                {
                    "scenario": scenario,
                    "horizon": int(h),
                    "variable": v,
                    "fcst_mean": float(fcst.mean(skipna=True)),
                    "fcst_std": fcst_std,
                    "realised_std": real_std,
                    "fcst_std_to_realised_std_ratio": ratio,
                }
            )
    return rows_corr_err, rows_clim_dist, rows_basic


if RUN_MODE in {"diagnostic_only", "smoke"}:
    print("\nComputing weather diagnostics by scenario ...")
    all_corr_err, all_clim_dist, all_basic = [], [], []
    for sc in SCENARIOS:
        if sc == "baseline_alpha_schedule":
            variant = pd.read_parquet(WEATHER_OPERATIONAL_BASELINE_PATH)
        else:
            variant_path = OPERATIONAL_VARIANT_PATHS[sc]
            if not variant_path.exists():
                print(f"  skipping {sc}: variant parquet missing.")
                continue
            variant = pd.read_parquet(variant_path)
        variant["target_date"] = pd.to_datetime(variant["target_date"]).dt.normalize()
        ce, cd, basic = diagnostic_rows_for(variant, sc, TRUTH_CLIM)
        all_corr_err.extend(ce)
        all_clim_dist.extend(cd)
        all_basic.extend(basic)
        del variant
        gc.collect()

    corr_err_df  = pd.DataFrame(all_corr_err)
    clim_dist_df = pd.DataFrame(all_clim_dist)
    basic_df     = pd.DataFrame(all_basic)
    # Include wet-day and cloud-regime diagnostics when columns exist.
    diag_df = corr_err_df.merge(
        clim_dist_df[["scenario","horizon","variable","mae_forecast_climatology","rmse_forecast_climatology"]],
        on=["scenario","horizon","variable"], how="left",
    ).merge(basic_df, on=["scenario","horizon","variable"], how="left")

    diag_df.to_csv(M2_ALPHA_SENS_DIR / "m2_alpha_sensitivity_weather_diagnostics.csv", index=False)
    corr_err_df.to_csv(M2_ALPHA_SENS_DIR / "m2_alpha_sensitivity_forecast_realised_correlation.csv", index=False)
    corr_err_df[["scenario","horizon","variable","n","mae_forecast_realised","rmse_forecast_realised"]] \
        .to_csv(M2_ALPHA_SENS_DIR / "m2_alpha_sensitivity_forecast_realised_error.csv", index=False)
    clim_dist_df.to_csv(M2_ALPHA_SENS_DIR / "m2_alpha_sensitivity_forecast_climatology_distance.csv", index=False)
    print(f"  saved 4 diagnostics CSVs under {project_relative(M2_ALPHA_SENS_DIR)}")


## Diagnostic figures

This optional section writes static diagnostic figures when matplotlib rendering is enabled and the run mode allows figure generation. The figures are descriptive outputs only and are not consumed by downstream model runs.


In [ ]:
try:
    import matplotlib
    # Headless rendering avoids GUI backend crashes on Windows.
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    _MPL_OK = True
except Exception:
    _MPL_OK = False

# Diagnostic figures are output-only PNGs and are not consumed by 09d.
# Rendering is guarded because matplotlib has caused native crashes on this machine.
RENDER_DIAGNOSTIC_FIGURES = True


if _MPL_OK and RENDER_DIAGNOSTIC_FIGURES and RUN_MODE in {"diagnostic_only", "smoke"} and not corr_err_df.empty:
    # Forecast-realised correlation by horizon and scenario.
    for v in VARIABLES:
        sub = corr_err_df.loc[corr_err_df["variable"] == v]
        if sub.empty:
            continue
        fig, ax = plt.subplots(figsize=(7.0, 4.2))
        for sc in SCENARIOS:
            s = sub.loc[sub["scenario"] == sc].sort_values("horizon")
            if s.empty:
                continue
            ax.plot(s["horizon"], s["corr_forecast_realised"], marker="o", label=sc)
        ax.set_xlabel("Horizon")
        ax.set_ylabel(f"Pearson corr ({v} forecast vs realised)")
        ax.set_title(f"Forecast-realised correlation: {v}")
        ax.set_ylim(-0.1, 1.05)
        ax.grid(True, alpha=0.3)
        ax.legend()
        fig.tight_layout()
        fig.savefig(FIG_DIR / f"forecast_realised_correlation_{v}.png", dpi=140)
        plt.close(fig)
    print(f"  saved forecast-realised correlation figures under {project_relative(FIG_DIR)}/")

    # Forecast-realised MAE by horizon and scenario.
    for v in VARIABLES:
        sub = corr_err_df.loc[corr_err_df["variable"] == v]
        if sub.empty:
            continue
        fig, ax = plt.subplots(figsize=(7.0, 4.2))
        for sc in SCENARIOS:
            s = sub.loc[sub["scenario"] == sc].sort_values("horizon")
            if s.empty:
                continue
            ax.plot(s["horizon"], s["mae_forecast_realised"], marker="o", label=sc)
        ax.set_xlabel("Horizon")
        ax.set_ylabel(f"MAE ({v} forecast vs realised)")
        ax.set_title(f"Forecast-realised MAE: {v}")
        ax.grid(True, alpha=0.3)
        ax.legend()
        fig.tight_layout()
        fig.savefig(FIG_DIR / f"forecast_realised_mae_{v}.png", dpi=140)
        plt.close(fig)

    # Forecast-climatology distance.
    if not clim_dist_df.empty:
        for v in VARIABLES:
            sub = clim_dist_df.loc[clim_dist_df["variable"] == v]
            if sub.empty:
                continue
            fig, ax = plt.subplots(figsize=(7.0, 4.2))
            for sc in SCENARIOS:
                s = sub.loc[sub["scenario"] == sc].sort_values("horizon")
                if s.empty:
                    continue
                ax.plot(s["horizon"], s["mae_forecast_climatology"], marker="o", label=sc)
            ax.set_xlabel("Horizon")
            ax.set_ylabel(f"MAE ({v} forecast vs climatology)")
            ax.set_title(f"Forecast-climatology distance: {v}")
            ax.grid(True, alpha=0.3)
            ax.legend()
            fig.tight_layout()
            fig.savefig(FIG_DIR / f"forecast_climatology_mae_{v}.png", dpi=140)
            plt.close(fig)

    # Alpha schedules.
    fig, ax = plt.subplots(figsize=(7.0, 4.2))
    for sc in SCENARIOS:
        s = alpha_schedules_df.loc[alpha_schedules_df["scenario"] == sc].sort_values("horizon")
        ax.plot(s["horizon"], s["alpha"], marker="o", label=f"{sc} (alpha(10)={SCENARIO_ALPHA_H10[sc]})")
    ax.set_xlabel("Horizon")
    ax.set_ylabel("alpha (realised-weight)")
    ax.set_title("Alpha schedules by sensitivity scenario")
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIG_DIR / "alpha_schedules.png", dpi=140)
    plt.close(fig)
    print(f"  saved alpha_schedules figure")
else:
    if not _MPL_OK:
        print("matplotlib unavailable; skipping figures.")
    elif not RENDER_DIAGNOSTIC_FIGURES:
        print("RENDER_DIAGNOSTIC_FIGURES=False; skipping diagnostic figure block (output-only PNGs, not needed for 09d).")


## M2-only ML sensitivity

Smoke and full modes fit tuned LightGBM M2 models for the alpha scenarios. Training uses rows with `target_date < origin_date` for the current horizon, and testing uses rows from the current forecast origin within the evaluation boundary. Existing M1 full-step metrics are used as the reference where available; no M1, M3 or M4 models are retrained for this sensitivity analysis.


In [ ]:
def alpha_ml_sensitivity():
    if not TUNING_BEST_PARAMS_PATH.exists():
        raise FileNotFoundError(
            f"Need tuned LightGBM params from notebook 08: {project_relative(TUNING_BEST_PARAMS_PATH)}"
        )

    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder
    try:
        from lightgbm import LGBMRegressor  # noqa: F401
    except Exception as exc:  # pragma: no cover
        raise ImportError(f"LightGBM not available: {type(exc).__name__}: {exc}")

    with open(TUNING_BEST_PARAMS_PATH, "r", encoding="utf-8") as f:
        tuning = json.load(f)
    static_kwargs = tuning.get("lightgbm_static_kwargs", {})
    static_kwargs.setdefault("objective", "regression")
    static_kwargs.setdefault("n_jobs", N_JOBS)
    static_kwargs.setdefault("random_state", RANDOM_STATE)
    static_kwargs.setdefault("verbose", -1)
    feature_set_params = {label: dict(tuning["feature_sets"][label]["best_params"]) for label in ML_FEATURE_SETS}
    for d in feature_set_params.values():
        d.pop("use_max_depth", None)
        d.setdefault("max_depth", -1)

    # Feature registry-driven feature sets, aligned with notebooks 07-09.
    registry = pd.read_csv(PROJECT_ROOT / "outputs" / "ml_panel" / "ml_panel_feature_registry.csv")
    registry.to_csv(M2_ALPHA_SENS_DIR / f"m2_alpha_sensitivity_feature_registry_snapshot{ML_FILE_SUFFIX}.csv", index=False)
    for col in [
        "allowed_m1_baseline",
        "allowed_m2_synthetic_point_weather",
        "allowed_m3_perfect_information",
        "allowed_m4_synthetic_engineered_weather",
        "leakage_risk",
        "known_at_origin",
    ]:
        registry[col] = registry[col].astype(str).str.lower().isin(["true", "1", "yes"])
    KEY_COLUMNS = registry.loc[registry["role"] == "key", "column"].tolist()
    TARGET_COLUMNS = registry.loc[
        registry["role"].isin(["target", "robustness_target"]),
        "column",
    ].tolist()
    FORBIDDEN = set(KEY_COLUMNS + TARGET_COLUMNS + ["origin_season"]) - {"AvdelingID", "Analyse_Kategori"}
    CAT_CANDS = [
        "AvdelingID",
        "AvdelingTekst",
        "Region",
        "Analyse_Kategori",
        "Ukedag",
        "HelligdagNavn",
        "season",
    ]

    def fset_cols(label):
        flag = {
            "M1": "allowed_m1_baseline",
            "M2": "allowed_m2_synthetic_point_weather",
            "M4": "allowed_m4_synthetic_engineered_weather",
        }[label]
        reg = registry.loc[registry[flag], "column"].tolist()
        reg = [c for c in reg if c not in FORBIDDEN]
        explicit = [c for c in CAT_CANDS if c in {"AvdelingID", "Analyse_Kategori"}]
        return list(dict.fromkeys(explicit + reg))

    feature_sets = {label: fset_cols(label) for label in ML_FEATURE_SETS}

    def split_num_cat(cols):
        cat = [c for c in cols if c in CAT_CANDS]
        num = [c for c in cols if c not in CAT_CANDS]
        return num, cat

    # Load only the ML rows and columns needed for the configured sensitivity run.
    panel = pd.read_parquet(
        ML_FORECAST_PANEL_PATH,
        columns=sorted(
            set(
                KEY_COLUMNS
                + ["Antall_total"]
                + [c for label in ML_FEATURE_SETS for c in feature_sets[label]]
            )
        ),
    )
    panel["target_date"] = pd.to_datetime(panel["target_date"]).dt.normalize()
    panel["origin_date"] = pd.to_datetime(panel["origin_date"]).dt.normalize()
    panel["horizon"] = panel["horizon"].astype("int8")
    panel = panel.loc[panel["horizon"].isin(ML_HORIZONS)].copy()
    panel = panel.loc[panel["Antall_total"].notna()].copy()

    eval_start = pd.Timestamp(EVAL_START_DATE).normalize()
    eval_end = pd.Timestamp(EVAL_END_DATE).normalize()

    def build_origin_schedule(panel_, eval_start_, eval_end_, mode, step_days, max_origins=None):
        available = sorted(
            pd.to_datetime(
                panel_.loc[
                    (panel_["origin_date"] >= eval_start_) & (panel_["origin_date"] <= eval_end_),
                    "origin_date",
                ]
                .dropna()
                .unique()
            ).normalize()
        )
        available_set = set(available)
        if mode == "daily":
            schedule_ = [d for d in pd.date_range(eval_start_, eval_end_, freq="D") if d in available_set]
        elif mode == "step":
            schedule_ = []
            cursor = eval_start_
            while cursor <= eval_end_:
                if cursor in available_set:
                    schedule_.append(cursor)
                cursor = cursor + pd.Timedelta(days=int(step_days))
        elif mode == "balanced_weekday":
            if max_origins is None:
                schedule_ = available
            else:
                counts = {i: 0 for i in range(7)}
                schedule_ = []
                for d in available:
                    wd = int(d.weekday())
                    if counts[wd] <= min(counts.values()) or len(schedule_) < 7:
                        schedule_.append(d)
                        counts[wd] += 1
                    if len(schedule_) >= max_origins:
                        break
        else:
            raise ValueError(f"Unsupported ORIGIN_SCHEDULE_MODE: {mode!r}")
        return schedule_[:int(max_origins)] if max_origins is not None else schedule_

    schedule = build_origin_schedule(panel, eval_start, eval_end, ORIGIN_SCHEDULE_MODE, ORIGIN_STEP_DAYS, ML_MAX_ORIGINS)
    print(f"ML origin schedule ({RUN_MODE}): {[d.date().isoformat() for d in schedule]}")

    def _weekday_label(series):
        return pd.to_datetime(series).dt.day_name()

    def _open_status_frame(df):
        if "is_open" in df.columns:
            return pd.Series(np.where(df["is_open"].astype("float64") > 0, "open", "closed"), index=df.index, name="open_status")
        if "Closed" in df.columns:
            return pd.Series(np.where(df["Closed"].astype("float64") > 0, "closed", "open"), index=df.index, name="open_status")
        return None

    def run_preflight_audit(panel_, schedule_):
        PREFLIGHT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)
        sched = pd.DataFrame({"origin_date": schedule_})
        if not sched.empty:
            sched["origin_weekday"] = _weekday_label(sched["origin_date"])
        sched["run_mode"] = RUN_MODE
        sched["origin_schedule_mode"] = ORIGIN_SCHEDULE_MODE
        sched["origin_step_days"] = ORIGIN_STEP_DAYS if ORIGIN_SCHEDULE_MODE == "step" else np.nan
        sched["n_origins"] = len(schedule_)
        sched["n_horizons"] = len(ML_HORIZONS)
        sched["n_feature_sets"] = len(ML_FEATURE_SETS)
        sched["n_scenarios"] = len(SCENARIOS)
        sched.to_csv(PREFLIGHT_AUDIT_DIR / f"origin_schedule_summary_{RUN_MODE}.csv", index=False)
        block = panel_.loc[
            panel_["origin_date"].isin(schedule_)
            & panel_["horizon"].isin(ML_HORIZONS)
            & (panel_["target_date"] <= eval_end)
        ].copy()
        warnings_list = []
        if block.empty:
            warnings_list.append("No evaluation rows found for the configured origin schedule.")
            target_weekday = pd.DataFrame(columns=["horizon", "target_weekday", "n_rows", "share"])
            origin_weekday = pd.DataFrame(columns=["origin_weekday", "n_origins", "share"])
            target_summary = pd.DataFrame(columns=["horizon", "n_rows", "mean_antall_total"])
            zero_sales = pd.DataFrame(columns=["horizon", "n_rows", "zero_sales_share"])
            category_dist = pd.DataFrame(columns=["horizon", "Analyse_Kategori", "n_rows", "share"])
            open_closed = pd.DataFrame()
        else:
            block["target_weekday"] = _weekday_label(block["target_date"])
            block["origin_weekday"] = _weekday_label(block["origin_date"])
            block["zero_sales"] = block["Antall_total"].astype("float64") <= 0
            target_weekday = block.groupby(["horizon", "target_weekday"], observed=True).size().reset_index(name="n_rows")
            target_weekday["share"] = target_weekday["n_rows"] / target_weekday.groupby(
                "horizon", observed=True
            )["n_rows"].transform("sum")
            origin_weekday = (
                sched.groupby("origin_weekday", observed=True).size().reset_index(name="n_origins")
                if "origin_weekday" in sched.columns
                else pd.DataFrame()
            )
            if not origin_weekday.empty:
                origin_weekday["share"] = origin_weekday["n_origins"] / origin_weekday["n_origins"].sum()
            target_summary = (
                block.groupby("horizon", observed=True)["Antall_total"]
                .agg(
                    n_rows="count",
                    mean_antall_total="mean",
                    median_antall_total="median",
                    std_antall_total="std",
                    min_antall_total="min",
                    max_antall_total="max",
                )
                .reset_index()
            )
            zero_sales = (
                block.groupby("horizon", observed=True)
                .agg(n_rows=("Antall_total", "size"), zero_sales_share=("zero_sales", "mean"))
                .reset_index()
            )
            category_dist = block.groupby(["horizon", "Analyse_Kategori"], observed=True).size().reset_index(name="n_rows")
            category_dist["share"] = category_dist["n_rows"] / category_dist.groupby(
                "horizon", observed=True
            )["n_rows"].transform("sum")
            status = _open_status_frame(block)
            if status is not None:
                block["open_status"] = status
                open_closed = block.groupby(["horizon", "open_status"], observed=True).size().reset_index(name="n_rows")
                open_closed["share"] = open_closed["n_rows"] / open_closed.groupby(
                    "horizon", observed=True
                )["n_rows"].transform("sum")
            else:
                open_closed = pd.DataFrame()
            min_wd = 5 if RUN_MODE == "full" else 3
            wd_n = (
                target_weekday.loc[target_weekday["n_rows"] > 0]
                .groupby("horizon", observed=True)["target_weekday"]
                .nunique()
            )
            for h in ML_HORIZONS:
                n_unique = int(wd_n.get(h, 0))
                if n_unique < min_wd:
                    warnings_list.append(
                        f"h={h} has only {n_unique} unique target weekdays; threshold={min_wd}."
                    )
            for h in [3, 10]:
                sub = target_weekday.loc[target_weekday["horizon"] == h]
                if not sub.empty:
                    sunday = float(sub.loc[sub["target_weekday"] == "Sunday", "share"].sum())
                    if sunday > 0.60:
                        warnings_list.append(f"h={h} target rows are Sunday-dominated: share={sunday:.3f}.")
                    if float(sub["share"].max()) >= 0.95:
                        warnings_list.append(
                            f"h={h} is effectively a single-weekday horizon: "
                            f"max_share={float(sub['share'].max()):.3f}."
                        )
            if not open_closed.empty:
                for h in [3, 10]:
                    sub = open_closed.loc[open_closed["horizon"] == h]
                    closed = float(sub.loc[sub["open_status"] == "closed", "share"].sum()) if not sub.empty else 0.0
                    if closed > 0.60:
                        warnings_list.append(f"h={h} target rows are closed-day dominated: share={closed:.3f}.")
            if not zero_sales.empty:
                zr = float(zero_sales["zero_sales_share"].max() - zero_sales["zero_sales_share"].min())
                if zr > 0.15:
                    warnings_list.append(f"Zero-sales share differs strongly across horizons: range={zr:.3f}.")
        target_weekday.to_csv(PREFLIGHT_AUDIT_DIR / f"target_weekday_distribution_by_horizon_{RUN_MODE}.csv", index=False)
        origin_weekday.to_csv(PREFLIGHT_AUDIT_DIR / f"origin_weekday_distribution_{RUN_MODE}.csv", index=False)
        target_summary.to_csv(PREFLIGHT_AUDIT_DIR / f"target_summary_by_horizon_{RUN_MODE}.csv", index=False)
        zero_sales.to_csv(PREFLIGHT_AUDIT_DIR / f"zero_sales_share_by_horizon_{RUN_MODE}.csv", index=False)
        category_dist.to_csv(PREFLIGHT_AUDIT_DIR / f"category_distribution_by_horizon_{RUN_MODE}.csv", index=False)
        if not open_closed.empty:
            open_closed.to_csv(PREFLIGHT_AUDIT_DIR / f"open_closed_distribution_by_horizon_{RUN_MODE}.csv", index=False)
        nfits = int(len(schedule_) * len(ML_HORIZONS) * len(ML_FEATURE_SETS) * len(SCENARIOS))
        hours = nfits * EST_MINUTES_PER_FIT / 60.0
        if hours > 36:
            warnings_list.append(f"Estimated runtime exceeds 36 hours: {hours:.2f} hours.")
        if (
            RUN_MODE == "full"
            and ORIGIN_SCHEDULE_MODE == "step"
            and ORIGIN_STEP_DAYS != 5
            and not ALLOW_NON_STEP5_FULL
        ):
            warnings_list.append("Full mode is configured with a non-step5 origin schedule.")
        hard = []
        if RUN_MODE == "full" and REQUIRE_PREFLIGHT_PASS_FOR_FULL:
            hard = [
                w for w in warnings_list
                if (
                    "unique target weekdays" in w
                    or "single-weekday" in w
                    or "non-step5" in w
                    or "No evaluation rows" in w
                )
            ]
        runtime = pd.DataFrame([
            {
                "run_mode": RUN_MODE,
                "origin_schedule_mode": ORIGIN_SCHEDULE_MODE,
                "origin_step_days": ORIGIN_STEP_DAYS if ORIGIN_SCHEDULE_MODE == "step" else np.nan,
                "n_origins": len(schedule_),
                "n_horizons": len(ML_HORIZONS),
                "n_feature_sets": len(ML_FEATURE_SETS),
                "n_scenarios": len(SCENARIOS),
                "estimated_n_fits": nfits,
                "est_minutes_per_fit": EST_MINUTES_PER_FIT,
                "estimated_runtime_hours": hours,
                "preflight_warning_count": len(warnings_list),
                "preflight_hard_fail_count": len(hard),
                "warnings": " | ".join(warnings_list),
                "hard_fail_reasons": " | ".join(hard),
            }
        ])
        runtime.to_csv(PREFLIGHT_AUDIT_DIR / f"estimated_runtime_{RUN_MODE}.csv", index=False)
        print(f"Pre-flight audit saved under {project_relative(PREFLIGHT_AUDIT_DIR)}")
        print(f"Estimated fits: {nfits:,}")
        print(f"Estimated runtime: {hours:.2f} hours at {EST_MINUTES_PER_FIT:.2f} minutes/fit")
        if warnings_list:
            print("Pre-flight warnings:")
            for w in warnings_list:
                print(f"  - {w}")
        else:
            print("Pre-flight warnings: none")
        return {
            "warnings": warnings_list,
            "hard_fail_reasons": hard,
            "passed_for_full": len(hard) == 0,
        }

    preflight_result = run_preflight_audit(panel, schedule)
    if RUN_MODE == "full" and REQUIRE_PREFLIGHT_PASS_FOR_FULL and not preflight_result["passed_for_full"]:
        raise AssertionError(
            "Full alpha-sensitivity ML stopped by pre-flight go/no-go check: "
            + " | ".join(preflight_result["hard_fail_reasons"])
        )
    if (
        RUN_MODE == "full"
        and ORIGIN_SCHEDULE_MODE == "step"
        and ORIGIN_STEP_DAYS != 5
        and not ALLOW_NON_STEP5_FULL
    ):
        raise AssertionError("Full alpha-sensitivity ML must use ORIGIN_STEP_DAYS = 5 unless ALLOW_NON_STEP5_FULL=True.")

    # Weather feature columns replaced by scenario-specific operational weather values.
    weather_point_cols = [POINT_COLUMNS[v] for v in VARIABLES]
    weather_uncert_cols = [
        c for c in panel.columns
        if any(c.endswith(suf) for suf in (
            "_fcst_std",
            "_fcst_p10",
            "_fcst_p90",
            "_fcst_interval_width",
        ))
    ]
    weather_aux_cols = [
        c for c in [
        "precip_amount_uncertainty", "precip_wet_amount_p90", "precip_wet_prob",
        "cloud_open_prob", "cloud_partly_prob", "cloud_overcast_prob",
        ]
        if c in panel.columns
    ]
    weather_replace_cols = weather_point_cols + weather_uncert_cols + weather_aux_cols

    def panel_with_scenario(scenario: str) -> pd.DataFrame:
        if scenario == "baseline_alpha_schedule":
            return panel
        variant_path = OPERATIONAL_VARIANT_PATHS[scenario]
        if not variant_path.exists():
            raise FileNotFoundError(f"Variant parquet missing: {variant_path}")
        variant_available = set(pq.ParquetFile(variant_path).schema_arrow.names)
        variant = pd.read_parquet(
            variant_path,
            columns=[
                "AvdelingID",
                "aggregation_window",
                "target_date",
                "horizon_days",
                "origin_date",
            ] + [c for c in weather_replace_cols if c in variant_available],
        )
        variant["target_date"] = pd.to_datetime(variant["target_date"]).dt.normalize()
        variant["origin_date"] = pd.to_datetime(variant["origin_date"]).dt.normalize()
        variant = variant.loc[variant["aggregation_window"] == "trade_08_19"].copy()
        variant = variant.rename(columns={"horizon_days": "horizon"})
        # Keep only configured sensitivity horizons.
        variant = variant.loc[variant["horizon"].isin(ML_HORIZONS)].copy()
        # Replace weather columns by left-joining scenario weather features.
        keep_keys = ["AvdelingID", "origin_date", "target_date", "horizon"]
        merge = variant[keep_keys + [c for c in weather_replace_cols if c in variant.columns]]
        out = (
            panel.drop(columns=[c for c in weather_replace_cols if c in panel.columns])
            .merge(merge, on=keep_keys, how="left")
        )
        return out

    def build_pipeline(num_cols, cat_cols, params):
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
        transformer = ColumnTransformer(
            transformers=[
                ("num", "passthrough", num_cols),
                ("cat", encoder, cat_cols),
            ],
            remainder="drop",
            sparse_threshold=0.3,
        )
        from lightgbm import LGBMRegressor
        full = {**static_kwargs, **params}
        return Pipeline([("preprocess", transformer), ("model", LGBMRegressor(**full))])

    def rmse(yt, yp):
        return float(np.sqrt(np.mean((np.asarray(yt, "float64") - np.asarray(yp, "float64")) ** 2)))

    def mae(yt, yp):
        return float(np.mean(np.abs(np.asarray(yt, "float64") - np.asarray(yp, "float64"))))

    def wape(yt, yp):
        denom = float(np.sum(np.abs(np.asarray(yt, "float64"))))
        if denom > 0:
            return float(np.sum(np.abs(np.asarray(yt, "float64") - np.asarray(yp, "float64"))) / denom)
        return float("nan")

    def bias(yt, yp):
        return float(np.mean(np.asarray(yp, "float64") - np.asarray(yt, "float64")))

    pred_frames = []
    metric_rows = []
    for scenario in SCENARIOS:
        try:
            scenario_panel = panel_with_scenario(scenario)
        except FileNotFoundError as exc:
            print(f"  skipping {scenario}: {exc}")
            continue
        for origin in schedule:
            train_panel = scenario_panel.loc[scenario_panel["target_date"] < origin]
            for label in ML_FEATURE_SETS:
                num_cols, cat_cols = split_num_cat(feature_sets[label])
                params = feature_set_params[label]
                pipeline = build_pipeline(num_cols, cat_cols, params)
                for h in ML_HORIZONS:
                    train_h = train_panel.loc[train_panel["horizon"] == h]
                    test_h = scenario_panel.loc[
                        (scenario_panel["origin_date"] == origin)
                        & (scenario_panel["target_date"] <= eval_end)
                        & (scenario_panel["horizon"] == h)
                    ]
                    if train_h.empty or test_h.empty:
                        continue
                    X_train = train_h[num_cols + cat_cols]
                    y_train = train_h["Antall_total"].astype("float32").to_numpy()
                    X_test = test_h[num_cols + cat_cols]
                    y_test = test_h["Antall_total"].astype("float32").to_numpy()
                    with warnings.catch_warnings():
                        warnings.simplefilter("ignore")
                        pipeline.fit(X_train, y_train)
                    y_pred = np.asarray(pipeline.predict(X_test), dtype="float64")
                    if CLIP_NEGATIVE_PREDICTIONS:
                        y_pred = np.clip(y_pred, 0.0, None)
                    pf = test_h[
                        ["AvdelingID", "Analyse_Kategori", "origin_date", "target_date", "horizon"]
                    ].copy()
                    pf["run_mode"] = RUN_MODE
                    pf["scenario"] = scenario
                    pf["feature_set"] = label
                    pf["y_true"] = y_test
                    pf["y_pred"] = y_pred
                    pf["absolute_error"] = np.abs(
                        pf["y_true"].astype("float64") - pf["y_pred"].astype("float64")
                    )
                    pf["squared_error"] = np.square(
                        pf["y_true"].astype("float64") - pf["y_pred"].astype("float64")
                    )
                    pred_frames.append(pf)
                    metric_rows.append({
                        "scenario": scenario,
                        "origin_date": origin.date().isoformat(),
                        "feature_set": label,
                        "horizon": int(h),
                        "n_train": int(len(train_h)),
                        "n_test": int(len(test_h)),
                        "rmse": rmse(y_test, y_pred),
                        "mae": mae(y_test, y_pred),
                        "wape": wape(y_test, y_pred),
                        "bias": bias(y_test, y_pred),
                    })
                    gc.collect()

    if pred_frames:
        pd.concat(pred_frames, ignore_index=True).to_parquet(
            M2_ALPHA_SENS_DIR / f"m2_alpha_sensitivity_predictions{ML_FILE_SUFFIX}.parquet",
            index=False,
        )
    metrics_smoke_df = pd.DataFrame(metric_rows)
    metrics_smoke_df.to_csv(M2_ALPHA_SENS_DIR / f"m2_alpha_sensitivity_metrics{ML_FILE_SUFFIX}.csv", index=False)

    # Gain summary and pairwise comparisons.
    if not metrics_smoke_df.empty:
        # M2 vs M1 and M4 vs M2, pooled across origins per scenario+horizon.
        pivot = (
            metrics_smoke_df.groupby(["scenario", "horizon", "feature_set"])[["rmse", "mae", "wape"]]
            .mean()
            .reset_index()
        )
        # M2 vs M1 reference comparison.
        m1 = pivot[pivot["feature_set"] == "M1"].set_index(["scenario", "horizon"])[["rmse", "mae", "wape"]]
        m2 = pivot[pivot["feature_set"] == "M2"].set_index(["scenario", "horizon"])[["rmse", "mae", "wape"]]
        m4 = pivot[pivot["feature_set"] == "M4"].set_index(["scenario", "horizon"])[["rmse", "mae", "wape"]]
        if m1.empty and M1_FULL_STEP5_METRICS_PATH.exists():
            m1_ref = pd.read_csv(M1_FULL_STEP5_METRICS_PATH)
            m1_ref = m1_ref.loc[
                (m1_ref["scope"].eq("feature_set_horizon"))
                & (m1_ref["feature_set"].eq("M1"))
                & (m1_ref["horizon"].isin(ML_HORIZONS)),
                ["horizon", "rmse", "mae", "wape"],
            ].copy()
            m1_blocks = []
            for sc in SCENARIOS:
                block = m1_ref.copy()
                block["scenario"] = sc
                m1_blocks.append(block)
            if m1_blocks:
                m1 = pd.concat(m1_blocks, ignore_index=True).set_index(
                    ["scenario", "horizon"]
                )[["rmse", "mae", "wape"]]
        m2_vs_m1 = (m1 - m2).reset_index()
        m2_vs_m1.columns = [
            "scenario",
            "horizon",
            "rmse_improvement_m2_vs_m1",
            "mae_improvement_m2_vs_m1",
            "wape_improvement_m2_vs_m1",
        ]
        m4_vs_m2 = (m2 - m4).reset_index()
        m4_vs_m2.columns = [
            "scenario",
            "horizon",
            "rmse_improvement_m4_vs_m2",
            "mae_improvement_m4_vs_m2",
            "wape_improvement_m4_vs_m2",
        ]
        gain = m2_vs_m1.merge(m4_vs_m2, on=["scenario", "horizon"], how="outer")
        gain.to_csv(M2_ALPHA_SENS_DIR / f"m2_alpha_sensitivity_gain_summary{ML_FILE_SUFFIX}.csv", index=False)
        m2_vs_m1.to_csv(M2_ALPHA_SENS_DIR / f"m2_alpha_sensitivity_m2_vs_m1_by_scenario{ML_FILE_SUFFIX}.csv", index=False)
        m4_vs_m2.to_csv(M2_ALPHA_SENS_DIR / f"m2_alpha_sensitivity_m4_vs_m2_by_scenario{ML_FILE_SUFFIX}.csv", index=False)
    print(f"Alpha ML sensitivity complete ({RUN_MODE}).")


if RUN_MODE in {"smoke", "full"}:
    print(f"\nRunning alpha ML sensitivity ({RUN_MODE}, step={ORIGIN_STEP_DAYS}) ...")
    alpha_ml_sensitivity()
else:
    print(f"\nRUN_MODE is {RUN_MODE!r}; skipping ML sensitivity.")


## Interpretation framework

This table records neutral interpretation cases for the alpha-sensitivity outputs. It is intended to guide thesis framing without pre-committing to a substantive conclusion.


In [ ]:
interpretation_rows = [
    {
        "case": 1,
        "pattern": "M2 still beats existing M1 reference under climatology_only_no_noise",
        "interpretation": (
            "Same-location, same-calendar-day normal weather has predictive value even without "
            "realised target-day anchoring or forecast noise."
        ),
    },
    {
        "case": 2,
        "pattern": (
            "M2 only beats existing M1 reference under baseline_alpha_schedule, not under "
            "stronger_climatology_drift / climatology_only_no_noise"
        ),
        "interpretation": (
            "Baseline long-horizon gains depend strongly on realised-weather anchoring; "
            "medium-range h=3-h=10 results should be framed as scenario analysis, not "
            "operational forecast evidence."
        ),
    },
    {
        "case": 3,
        "pattern": "climatology_only_with_noise differs materially from climatology_only_no_noise",
        "interpretation": (
            "Calibrated forecast-like noise affects the demand signal; treat the noisy version "
            "as robustness/stress-test, not the main low-information benchmark."
        ),
    },
    {
        "case": 4,
        "pattern": "M2 remains stable across all scenarios",
        "interpretation": "Point weather information is robust to the climatology-drift prior.",
    },
    {
        "case": 5,
        "pattern": "h=0-h=2 remains strong while h=3-h=10 collapses under climatology_only_no_noise",
        "interpretation": (
            "Operational evidence is concentrated in actual archived MEPS horizons; "
            "medium-range results should be framed as sensitivity / scenario evidence."
        ),
    },
]
interp_df = pd.DataFrame(interpretation_rows)
interp_df.to_csv(M2_ALPHA_SENS_DIR / "m2_alpha_sensitivity_interpretation_framework.csv", index=False)
print(f"Saved: {project_relative(M2_ALPHA_SENS_DIR / 'm2_alpha_sensitivity_interpretation_framework.csv')}")


## Manual run instructions

Choose `RUN_MODE` deliberately before execution. Use `notebook_only` for static validation, `diagnostic_only` for weather diagnostics without ML fitting, `smoke` for a small M2-only model check, and `full` only when ready to run the complete h=3..10 M2 sensitivity over the configured origin schedule. Inspect the diagnostic CSVs before relying on the ML outputs.
